# **4 Кейс**

**Описание кейса:** требуется классифицировать риск задержки доставки заказа более чем на 5 минут от обещанного времени на этапе оформления, используя параметры ресторана, рейтинг курьера, погодные условия и пробки.

https://www.kaggle.com/datasets/razanihababdellatif/food-delivery-orders-and-eta-logistics-dataset

## **Импорт библиотек**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## **Загрузка данных**

In [ ]:
df = pd.read_csv("food_delivery_orders_dataset.csv")

## **Первичный анализ**

In [ ]:
pd.set_option('display.max_columns', None)
df.head()

Целевой показатель - **late_delivery**  
Заказ помечается как доставленный с опозданием (late_delivery = 1.0) только в том случае, если фактическое время доставки превышает расчетное более чем на 5 минут

Явно лишние показатели:  
**order_id**, **customer_id**, **restaurant_id** - не несут смысла  
**actual_delivery_time_minutes** - фактическое время доставки не имеет значения, важен факт опоздал курьер или нет

Нужно проверить как **late_delivery** зависит от **driver_id**  

Показатели **order_date**, **order_hour**, **day_of_week**, **is_weekend** можно выразить из **order_timestamp**  
Показатель **city** можно выразить из **delivery_area**.  
Показатель **order_total** выражается из суммы показателей (**subtotal** - **discount_percent**%), **tax_amount**, **service_fee**, **delivery_fee**

In [ ]:
df['late_delivery'].value_counts()

Есть перевес в сторону записей без опоздания. Записей без опоздания больше примерно в 1.73 раза

In [ ]:
print(f"Кол-во строк: {df.shape[0]}")
print(f"Кол-во столбцов: {df.shape[1]}")

In [ ]:
df.dtypes.value_counts()

Категориальных параметров - 17  
Непрерывных параметров - 20 (12 + 8)

In [ ]:
# Не получилось вывести все колонки через df.info ))
pd.DataFrame({
    "dtype":   df.dtypes,
    "non_null": df.notna().sum(),
    "null":    df.isna().sum(),
    "null_%":  (df.isna().mean() * 100).round(2),
})

Есть несоответствие типов:  
**order_timestamp** - строка, хотя логичней тип datetime.datetime  
**order_date** - строка, хотя логичней тип datetime.date

## **Анализ непрерывных показателей**

In [ ]:
df.describe().T

Аномалий не вижу  
Скорей всего есть выбросы, если смотреть на 75 квартиль и максимальное значение параметров **order_total**, **service_fee**, **restaurant_preparation_time_minutes** и **tip_amount**. Посмотрим на этой подробней при визуализации

## **Анализ категориальных показателей**

In [ ]:
df.describe(include=["str"]).T

Аномалий не вижу

## **Анализ дублей**

In [ ]:
df.duplicated().sum()

Дублей нет

## **Анализ пропусков**

**tip_amount**, **actual_delivery_time_minutes**, **late_delivery**, **customer_rating** имеют одинаковое количество пропусков - 868  
У **cancellation_reason** 49132 пропусков, что равно 50000-868  

**late_delivery** - целевая параметр, в ней есть пропуски. Мы можем удалить эти строки, так как нам не нужны строки без ответа

Можно предположить что все пропуски в одних и тех же строках, и после очистки строк с пустым **late_delivery** пропуски пропадут  

Проверим строки с пустым **late_delivery**:

In [ ]:
df[df["late_delivery"].isna()].head(15)

Для записей у которых **late_delivery** пустой:  
**tip_amount** = NaN  
**actual_delivery_time_minutes** = NaN  
**order_status** = Cancelled  
**cancellation_reason** заполнен  
**customer_rating** = NaN  

Из этого можно сделать вывод: **customer_rating**, **actual_delivery_time_minutes** и **tip_amount** заполняются только после выполнения доставки. Для прогноза будет ли выполнена доставка без опоздания они НЕ нужны  
**cancellation_reason** заполнен только для записей у которых **late_delivery** не заполнен, для обучения нам нужны только записи с заполненным **late_delivery**, этот параметр не нужен  
**order_status** имеет 2 значения **Cancelled** (для записей **late_delivery.isna()**) и **Completed** (для записей **late_delivery.notna()**). Нас интересуют только записи с заполненным **late_delivery**, в этих записях у order_status будет 1 уникальное значение - **Completed**, можно удалить этот параметр

## **Промежуточный итог**

**Размер датасета:**  
Кол-во строк: 50000  
Кол-во столбцов: 37  

Категориальных параметров - 17  
Непрерывных параметров - 20  

Целевой показатель - **late_delivery**  
Есть перевес в сторону записей без опоздания (**late_delivery** = 0). Записей без опоздания больше примерно в 1.73 раза  

Есть несоответствие типов:  
**order_timestamp** - строка, хотя логичней тип datetime.datetime  
**order_date** - строка, хотя логичней тип datetime.date. 

**Дубликатов в датасете нет**

**Пропуски:**
**tip_amount**, **actual_delivery_time_minutes**, **customer_rating** и **late_delivery** имеют одинаковое количество пропусков - 868  
У **cancellation_reason** 49132 пропусков, что равно 50000-868. Там где **late_delivery** = NaN - **cancellation_reason** имеет значение, во всех остальных случаях **cancellation_reason** пустой  

Есть записи с пустой целевой переменной (**late_delivery** = NaN). Нужно будет удалить эти записи из Датасета
Все пропуски в одних и тех же строках, и после очистки строк с пустым **late_delivery** пропуски пропадут. Останутся пропуски в **cancellation_reason**, но об этом ниже  

На этом этапе уже можно выделить **лишние показатели**:  
**order_id**, **customer_id**, **restaurant_id** - не несут смысла для прогноза будет ли доставка выполнена без опоздания  
**actual_delivery_time_minutes** - фактическое время доставки в этой задаче не имеет значения, важен факт опоздал курьер или нет  
**customer_rating**, **actual_delivery_time_minutes** и **tip_amount** заполняются только после выполнения доставки. Для прогноза будет ли доставка выполнена без опоздания они не нужны  
**cancellation_reason** заполнен только для записей у которых **late_delivery** не заполнен, для обучения нам нужны только записи с заполненным **late_delivery**, этот параметр не нужен  
**order_status** имеет 2 значения **Cancelled** (для записей **late_delivery** = NaN) и **Completed** (для всех остальных записей). Нас интересуют только записи с заполненным **late_delivery**, в этих записях у **order_status** будет 1 уникальное значение - **Completed**, можно удалить этот параметр. 

Итого список столбцов на удаление: **order_id**, **customer_id**, **restaurant_id**, **actual_delivery_time_minutes**, **customer_rating**, **tip_amount**, **cancellation_reason**, **order_status**


**Открытые вопросы после первичного анализа:**  
1. Есть ли зависимоть **late_delivery** от **driver_id**?  
2. Показатели **order_date**, **order_hour**, **day_of_week**, **is_weekend** можно выразить из **order_timestamp**. Какие признаки оставить?  
3. Показатель **city** можно выразить из **delivery_area**. Какой признаки оставить?  
4. Показатель **order_total** выражается из суммы показателей (**subtotal** - **discount_percent**%), **tax_amount**, **service_fee**, **delivery_fee**. Какие признаки оставить?  


## **Преобразования после первичного анализа**

Удаление строк с пустым целевым показателем:

In [ ]:
df = df[df["late_delivery"].notna()]

Преобразование показателей с датой к типу datetime и date:

In [ ]:
df['order_timestamp'] = pd.to_datetime(df['order_timestamp'])
df['order_date'] = pd.to_datetime(df['order_date']).dt.date

Удаление лишних показателей:

In [ ]:
df.drop([
    "order_id",
    "customer_id",
    "restaurant_id",
    "actual_delivery_time_minutes",
    "customer_rating",
    "tip_amount",
    "cancellation_reason",
    "order_status",
], axis=1, inplace=True)

Проверка текущего размера датасета и пропусков:

In [ ]:
print(f"Кол-во строк: {df.shape[0]}")
print(f"Кол-во столбцов: {df.shape[1]}")

pd.DataFrame({
    "dtype":   df.dtypes,
    "null":    df.isna().sum()
})

## **Анализ выбросов**

In [ ]:
numeric_columns = [
    'order_hour',
    'is_weekend',
    'customer_age',
    'restaurant_rating',
    'items_count',
    'subtotal',
    'discount_percent',
    'tax_amount',
    'service_fee',
    'delivery_fee',
    'order_total',
    'distance_km',
    'delivery_partner_experience_months',
    'delivery_partner_rating',
    'restaurant_preparation_time_minutes',
    'estimated_delivery_time_minutes',
]

fig, axes = plt.subplots(4, 4, figsize=(15, 15))
axes = axes.flatten()

for idx, col in enumerate(numeric_columns):
    ax = axes[idx]
    df.boxplot(column=col, ax=ax)
    ax.set_xlabel('')
    ax.set_title(col, fontsize=12, fontweight='bold')

plt.suptitle('Выбросы по показателям', fontsize=16, fontweight='bold', y=.94)
plt.subplots_adjust(
    wspace=0.2,
    hspace=0.3
)
plt.show()

**Вывод:**  

## **Одномерный анализ**

In [ ]:
numeric_columns = [
    'order_hour',
    'is_weekend',
    'customer_age',
    'restaurant_rating',
    'items_count',
    'distance_km',
    'delivery_partner_experience_months',
    'delivery_partner_rating',
    'restaurant_preparation_time_minutes',
    'estimated_delivery_time_minutes',
]

fig, axes = plt.subplots(5, 2, figsize=(15, 20))
axes = axes.flatten()

for idx, col in enumerate(numeric_columns):
    ax = axes[idx]
    sns.histplot(
        data=df,
        x=col,
        hue='late_delivery',
        bins="auto",
        multiple='dodge',
        ax=ax
    )
    ax.set_xlabel('')
    ax.set_title(col, fontsize=12, fontweight='bold')

plt.suptitle('Непрерывные показатели', fontsize=16, fontweight='bold', y=.94)
plt.subplots_adjust(
    wspace=0.2,
    hspace=0.4
)
plt.show()

### % Опозданий в зависимости от расстояния

In [ ]:
distance_km = df.groupby("distance_km").agg(
    orders_count=("distance_km", "count"),
    late_delevery_percent=("late_delivery", "mean")
).sort_values(by="orders_count", ascending=False).reset_index()

plt.figure(figsize=(12, 7))

sns.lineplot(
    data=distance_km,
    x="distance_km",
    y="late_delevery_percent",
)

ax.set_xlabel('')
ax.set_title(col, fontsize=12, fontweight='bold')

plt.show()

### % Опозданий в зависимости от времени заказа

In [ ]:
order_hour = df.groupby("order_hour").agg(
    orders_count=("order_hour", "count"),
    late_delevery_percent=("late_delivery", "mean")
).sort_values(by="orders_count", ascending=False).reset_index()

print(f"Разница между максимальным и минимальным % опозданий: {order_hour["late_delevery_percent"].max() - order_hour["late_delevery_percent"].min():.2f}")

plt.figure(figsize=(12, 7))

sns.barplot(
    data=order_hour,
    x="order_hour",
    y="late_delevery_percent",
    legend=False
)

ax.set_xlabel('')
ax.set_title(col, fontsize=12, fontweight='bold')

plt.show()

### % Опозданий в зависимости от времени приготовления

In [ ]:
restaurant_preparation_time_minutes = df.groupby("restaurant_preparation_time_minutes").agg(
    orders_count=("restaurant_preparation_time_minutes", "count"),
    late_delevery_percent=("late_delivery", "mean")
).sort_values(by="orders_count", ascending=False).reset_index()

print(f"Разница между максимальным и минимальным % опозданий: {restaurant_preparation_time_minutes["late_delevery_percent"].max() - restaurant_preparation_time_minutes["late_delevery_percent"].min():.2f}")

plt.figure(figsize=(18, 7))

sns.barplot(
    data=restaurant_preparation_time_minutes,
    x="restaurant_preparation_time_minutes",
    y="late_delevery_percent",
    legend=False
)

ax.set_xlabel('')
ax.set_title(col, fontsize=12, fontweight='bold')

plt.show()

In [ ]:
# numeric_columns = [
#     'subtotal',
#     'discount_percent',
#     'tax_amount',
#     'service_fee',
#     'delivery_fee',
#     'order_total',
# ]

# fig, axes = plt.subplots(6, 1, figsize=(15, 20))
# axes = axes.flatten()

# for idx, col in enumerate(numeric_columns):
#     ax = axes[idx]
#     sns.histplot(
#         data=df,
#         x=col,
#         hue='late_delivery',
#         bins="auto",
#         multiple='dodge',
#         ax=ax
#     )
#     ax.set_xlabel('')
#     ax.set_title(col, fontsize=12, fontweight='bold')

# plt.suptitle('Финансовые показатели', fontsize=16, fontweight='bold', y=.94)
# plt.subplots_adjust(
#     wspace=0.2,
#     hspace=0.4
# )
# plt.show()

In [ ]:
str_columns = [
    'day_of_week',
    'city',
    'customer_type',
    'restaurant_type',
    'restaurant_primary_category',
    'payment_method',
    'weather',
    'traffic_level',
]

fig, axes = plt.subplots(4, 2, figsize=(15, 20))
axes = axes.flatten()

for idx, col in enumerate(str_columns):
    ax = axes[idx]
    sns.histplot(
        data=df,
        x=col,
        hue='late_delivery',
        bins="auto",
        multiple='dodge',
        ax=ax
    )
    ax.set_xlabel('')
    ax.set_title(col, fontsize=12, fontweight='bold')

plt.suptitle('Категориальные показатели', fontsize=16, fontweight='bold', y=.94)
plt.subplots_adjust(
    wspace=0.2,
    hspace=0.4
)
plt.show()

### % Опозданий в пробки

In [ ]:
traffic_level = df.groupby("traffic_level").agg(
    orders_count=("traffic_level", "count"),
    late_delevery_percent=("late_delivery", "mean")
).sort_values(by="orders_count", ascending=False).reset_index()

print(f"Разница между максимальным и минимальным % опозданий: {traffic_level["late_delevery_percent"].max() - traffic_level["late_delevery_percent"].min():.2f}")

plt.figure(figsize=(12, 7))

sns.barplot(
    data=traffic_level.head(25),
    y="traffic_level",
    x="late_delevery_percent",
    legend=False
)
ax.set_xlabel('')
ax.set_title(col, fontsize=12, fontweight='bold')

plt.show()

### % Опозданий в погоду

In [ ]:
weather = df.groupby("weather").agg(
    orders_count=("weather", "count"),
    late_delevery_percent=("late_delivery", "mean")
).sort_values(by="orders_count", ascending=False).reset_index()

print(f"Разница между максимальным и минимальным % опозданий: {weather["late_delevery_percent"].max() - weather["late_delevery_percent"].min():.2f}")

plt.figure(figsize=(12, 7))

sns.barplot(
    data=weather.head(25),
    y="weather",
    x="late_delevery_percent",
    legend=False
)
ax.set_xlabel('')
ax.set_title(col, fontsize=12, fontweight='bold')

plt.show()

### % Опозданий доставщиков

In [ ]:
top_drivers = df.groupby("driver_id").agg(
    orders_count=("driver_id", "count"),
    late_delevery_percent=("late_delivery", "mean")
).sort_values(by="orders_count", ascending=False).reset_index()

print(f"Разница между максимальным и минимальным % опозданий: {top_drivers["late_delevery_percent"].max() - top_drivers["late_delevery_percent"].min():.2f}")

plt.figure(figsize=(12, 7))

sns.barplot(
    data=top_drivers.head(25),
    y="driver_id",
    x="late_delevery_percent",
    legend=False
)
ax.set_xlabel('')
ax.set_title(col, fontsize=12, fontweight='bold')

plt.show()

### % Опозданий в район

In [ ]:
area = df.groupby("delivery_area").agg(
    orders_count=("delivery_area", "count"),
    late_delevery_percent=("late_delivery", "mean")
).sort_values(by="orders_count", ascending=False).reset_index()

print(f"Разница между максимальным и минимальным % опозданий: {area["late_delevery_percent"].max() - area["late_delevery_percent"].min():.2f}")

plt.figure(figsize=(12, 7))

sns.barplot(
    data=area.head(25),
    y="delivery_area",
    x="late_delevery_percent",
    legend=False
)
ax.set_xlabel('')
ax.set_title(col, fontsize=12, fontweight='bold')

plt.show()

### % Опозданий от дня недели

In [ ]:
day_of_week = df.groupby("day_of_week").agg(
    orders_count=("day_of_week", "count"),
    late_delevery_percent=("late_delivery", "mean")
).sort_values(by="orders_count", ascending=False).reset_index()

print(f"Разница между максимальным и минимальным % опозданий: {day_of_week["late_delevery_percent"].max() - day_of_week["late_delevery_percent"].min():.2f}")

plt.figure(figsize=(12, 7))

sns.barplot(
    data=day_of_week.head(25),
    y="day_of_week",
    x="late_delevery_percent",
    legend=False
)
ax.set_xlabel('')
ax.set_title(col, fontsize=12, fontweight='bold')

plt.show()

## **Многомерный анализ**

In [ ]:
plt.figure(figsize=(14, 10))

sns.heatmap(
    df.select_dtypes(include='number').corr(),
    annot=True,
    cmap='coolwarm',
    center=0,
    fmt='.2f'
)

plt.title('Корреляционная матрица')
plt.show()

У **order_total** высокая зависимость от **subtotal**, **tax_amount**, **service_fee**, потому что **order_total** это сумма этих показателей. Можно удалить **subtotal**, **tax_amount**, **service_fee**  
Также у **order_total** зависимость ближе к высокой от **items_count**. Нужно оставить 1 из них, оставим **order_total**  
У **restaurant_preparation_time_minutes** высокая зависимость от **order_total**. Нужно оставить 1 из них, та как нам важно время доставки, оставим restaurant_preparation_time_minutes  
У **distance_km** высокая зависимость от **estimated_delivery_time_minutes**. Видимо ожидаемое время доставки считается от дистанции. Оставим дистанцию

In [ ]:
# Pairplot
# sns.pairplot(df, diag_kind='kde',
#              plot_kws={'alpha': 0.5, 's': 10},
#              diag_kws={'linewidth': 2})
# plt.suptitle('Pairplot: Взаимосвязи между ключевыми характеристиками',
#              fontsize=14, fontweight='bold', y=1.02)
# plt.show()